# MQTT Publisher 測試程式

這個 notebook 示範如何使用 `paho-mqtt` 建立一個簡單的 MQTT 發布者（Publisher）應用程式，用來做功能測試。


In [ ]:
import time
import json
import random

import paho.mqtt.client as mqtt

# MQTT Broker 設定（預設使用公開測試 Broker）
BROKER_HOST = "test.mosquitto.org"  # 你也可以改成自己的 Broker IP，例如 "192.168.1.10"
BROKER_PORT = 1883
TOPIC = "test/raspberry_pi_pico"    # 測試主題（建議改成自己專用的路徑）
CLIENT_ID = f"publisher_test_{random.randint(1000, 9999)}"  # 隨機客戶端 ID 避免衝突

print("MQTT Publisher 測試設定完成")
print(f"Broker: {BROKER_HOST}:{BROKER_PORT}")
print(f"Topic: {TOPIC}")
print(f"Client ID: {CLIENT_ID}")


In [ ]:
# 建立 MQTT 客戶端並連線到 Broker

client = mqtt.Client(client_id=CLIENT_ID)

connected = False

# 連線回呼函數

def on_connect(client, userdata, flags, rc):
    global connected
    if rc == 0:
        connected = True
        print("✓ 成功連接到 MQTT Broker")
        print(f"  - 連線標誌: {flags}")
    else:
        connected = False
        error_messages = {
            1: "協議版本不正確",
            2: "客戶端 ID 無效",
            3: "伺服器不可用",
            4: "用戶名或密碼錯誤",
            5: "未授權",
        }
        error_msg = error_messages.get(rc, "未知錯誤")
        print(f"✗ 連接失敗，錯誤代碼 {rc}: {error_msg}")


# 發布成功回呼函數

def on_publish(client, userdata, mid):
    print(f"✓ 消息已發布 (Message ID: {mid})")


client.on_connect = on_connect
client.on_publish = on_publish

print(f"\n正在連接到 {BROKER_HOST}:{BROKER_PORT}...")
try:
    client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    client.loop_start()  # 開始背景處理
    time.sleep(2)        # 留一點時間讓連線完成

    if connected:
        print("✓ 準備就緒，可以開始發布消息")
    else:
        print("⚠ 連線狀態未知，請檢查網路或 Broker 設定")
except Exception as e:
    print(f"✗ 連接錯誤: {e}")
    print("  請檢查：")
    print("  1. 網路連線是否正常")
    print("  2. Broker 地址是否正確")
    print("  3. 防火牆是否阻擋了連接")


In [ ]:
# 發布一則簡單文字消息

message = "Hello MQTT!"
result = client.publish(TOPIC, message, qos=1)

if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"發布文字消息成功: {message}")
else:
    print(f"發布失敗，錯誤代碼: {result.rc}")


In [ ]:
# 發布一則 JSON 格式的測試消息

payload = {
    "sensor": "temperature",
    "value": 25.5,
    "unit": "celsius",
    "timestamp": time.time(),
}

json_message = json.dumps(payload)
result = client.publish(TOPIC, json_message, qos=1)

if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"發布 JSON 消息成功: {json_message}")
else:
    print(f"發布失敗，錯誤代碼: {result.rc}")


In [ ]:
# 連續發布多條測試消息

for i in range(5):
    message = f"消息 #{i + 1}: 當前時間 {time.strftime('%Y-%m-%d %H:%M:%S')}"
    result = client.publish(TOPIC, message, qos=1)
    if result.rc == mqtt.MQTT_ERR_SUCCESS:
        print(f"✓ {message}")
    else:
        print(f"✗ 發布失敗（第 {i + 1} 條），錯誤代碼: {result.rc}")
    time.sleep(1)  # 每秒發布一條消息


In [ ]:
# 關閉 MQTT 連線

client.loop_stop()  # 停止背景處理
client.disconnect()
print("✓ MQTT 連線已關閉")


## 使用方式說明

1. 依序執行上面的 cells：
   - Cell 1：匯入套件並設定 Broker / Topic / Client ID
   - Cell 2：建立 MQTT 客戶端並連線
   - Cell 3：發送一則簡單文字消息
   - Cell 4：發送一則 JSON 消息
   - Cell 5：連續發送多條測試消息
   - Cell 6：關閉連線

2. 若要改用「自己的 MQTT Broker」：
   - 在 Cell 1 中把 `BROKER_HOST` 改成你的 Broker IP 或主機名，例如：`"192.168.1.10"`
   - 若連接埠不是 1883，就把 `BROKER_PORT` 改成正確的 port
   - 把 `TOPIC` 改成你想用的測試主題，例如：`"home/test"`

3. 測試是否有收到消息：
   - 可以在另一台電腦或同一台機器上，用 `mosquitto_sub` 或圖形化工具（例如 MQTTX）訂閱同一個主題，確認有收到上面發佈的內容。
